<a href="https://colab.research.google.com/github/ibnu-aulia/arabic-text-analyzer/blob/main/arabic_tokenization_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arabic Tokenization Experiments

I tested several ways to split Arabic text into tokens, from plain `split()` to AraBERT, Stanza, and my own splitter. For each one I looked at where it works and where it breaks.

Sections: 1) manual tokenization, 2) Arabic vs English, 3) mini stemmer, 4) TF-IDF from scratch, 5) dialects and ambiguity, 6) AraBERT tokenizers, 7) Stanza, 8) my own و splitter, 9) summary.

## 1. Manual tokenization

In [1]:
import re

text1 = "ذهب الولد إلى المدرسة وقرأ الكتاب"
text2 = "ذهب الولد، ثم عاد!"

tokens1 = text1.split()
print(len(tokens1))
print(tokens1)

tokens2 = text2.split()
print(len(tokens2))
print(tokens2)

# Keep only Arabic letters, so punctuation is dropped
tokens3 = re.findall(r"[\u0621-\u064A]+", text2)
print(len(tokens3))
print(tokens3)

6
['ذهب', 'الولد', 'إلى', 'المدرسة', 'وقرأ', 'الكتاب']
4
['ذهب', 'الولد،', 'ثم', 'عاد!']
4
['ذهب', 'الولد', 'ثم', 'عاد']


## 2. Arabic vs English

In [2]:
arab1 = "وقرأ الولد الكتاب"
eng1 = "and the boy read the book"
arab2 = "سيكتبونها"
eng2 = "they will write it"

print(len(arab1.split()))
print(len(eng1.split()))
print(len(arab2.split()))
print(len(eng2.split()))

arab_total = len(arab1.split()) + len(arab2.split())
eng_total = len(eng1.split()) + len(eng2.split())
print(arab_total, eng_total)

3
6
1
4
4 10


## 3. Mini stemmer

In [3]:
def simple_stem(word):
    if word.startswith("ال") and len(word) > 3:
        word = word[2:]
    elif word.startswith("و") and len(word) > 3:
        word = word[1:]
    return word

words = ["الولد", "وقرأ", "المدرسة", "ولد", "والد"]

for w in words:
    print(w, simple_stem(w))

الولد ولد
وقرأ قرأ
المدرسة مدرسة
ولد ولد
والد الد


## 4. TF-IDF from scratch

In [4]:
import math
from sklearn.feature_extraction.text import TfidfVectorizer

docs = ["الطعام لذيذ جدا", "الخدمة سيئة جدا", "الطعام سيئ"]
N = len(docs)
tokenized = [d.split() for d in docs]

# Manual TF-IDF for the first sentence
for word in tokenized[0]:
    tf = tokenized[0].count(word)
    doc_freq = sum(1 for t in tokenized if word in t)
    idf = math.log((1 + N) / (1 + doc_freq)) + 1
    print(word, doc_freq, round(idf, 4), round(tf * idf, 4))

# Same calculation with scikit-learn, for comparison
vec = TfidfVectorizer(norm=None)
X = vec.fit_transform(docs)
print(dict(zip(vec.get_feature_names_out(), X[0].toarray()[0].round(4))))

الطعام 2 1.2877 1.2877
لذيذ 1 1.6931 1.6931
جدا 2 1.2877 1.2877
{'الخدمة': np.float64(0.0), 'الطعام': np.float64(1.2877), 'جدا': np.float64(1.2877), 'سيئ': np.float64(0.0), 'سيئة': np.float64(0.0), 'لذيذ': np.float64(1.6931)}


## 5. Dialects and ambiguity

In [5]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("hadyelsahar/ar_res_reviews", split="train")
df = pd.DataFrame(ds)

words = ["بس", "وايد", "مرة", "زين", "حلو"]

for w in words:
    count_sub = df["text"].str.contains(w).sum()                       # letters anywhere in the review
    count_tok = df["text"].apply(lambda t: w in t.split()).sum()       # whole word only
    print(w, count_sub, count_tok)

README.md:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.89MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8364 [00:00<?, ? examples/s]

بس 1950 1272
وايد 15 14
مرة 631 516
زين 171 51
حلو 1069 498


**What the numbers show**

Counting letter sequences (`count_sub`) also matches the same letters inside other words, so it overcounts. Counting whole tokens (`count_tok`) can miss forms with attached prefixes or endings. Neither method is perfect without morphology.

`مرة` is ambiguous: in Modern Standard Arabic it usually means "once / one time", while in dialect it often means "very". Counting cannot tell these two meanings apart.

## 6. AraBERT tokenizers

### 6a. `arabertv2` on raw text

This model expects text that was pre-segmented with Farasa. I gave it raw text, so the subwords are broken (for example `الو` + `##لد`).

In [6]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
text = "وقرأ الولد الكتاب"

tokens = tok.tokenize(text)
print(len(tokens))
print(tokens)

ids = tok(text)["input_ids"]
print(len(ids))   # tokens + 2 special tokens
print(ids)

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/720k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

6
['وقر', '##أ', 'الو', '##لد', 'الكت', '##اب']
8
[33, 24121, 225, 25598, 8062, 21507, 305, 34]


### 6b. `arabertv02` with its preprocessor

This version does not need Farasa. Common words stay whole, but the و is not separated.

In [7]:
!pip install arabert

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for emoji: filename=emoji-1.4.2-py3-none-any.whl size=186539 sha256=c82e34015345f5a828406a5b6d275e25296d92de6622c469ba61e4a24e8df009
  Stored in directory: /root/.cache/pip/wheels/e1/79/2a/9864bac03173079abcc089d3eb3ec87456ac09c381b801f1a3
Successfully built emoji
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [arabert]


In [8]:
from arabert.preprocess import ArabertPreprocessor
from transformers import AutoTokenizer

name = "aubmindlab/bert-base-arabertv02"
prep = ArabertPreprocessor(model_name=name)
tok = AutoTokenizer.from_pretrained(name)

texts = ["وقرأ الولد الكتاب", "والد الولد"]

for text in texts:
    clean = prep.preprocess(text)
    tokens = tok.tokenize(clean)
    print(text)
    print(len(tokens), tokens)

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

وقرأ الولد الكتاب
3 ['وقرأ', 'الولد', 'الكتاب']
والد الولد
2 ['والد', 'الولد']


### 6c. Farasa (failed)

I tried `arabertv2` with Farasa pre-segmentation. The Farasa download did not return a zip file, and the code silently produced 0 tokens (`0 []`) instead of raising a clear error. I removed that code so that "Restart and run all" works.

## 7. Stanza

Stanza can split clitics such as the و. It separates it in `وقرأ`, but it also breaks `والد` ("father") into `و` + `ألد`.

In [9]:
!pip install stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 822.5/822.5 kB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [stanza]


In [10]:
import stanza

stanza.download("ar")
nlp = stanza.Pipeline("ar", processors="tokenize,mwt")

texts = ["وقرأ الولد الكتاب", "والد الولد"]

for text in texts:
    doc = nlp(text)
    for sent in doc.sentences:
        for token in sent.tokens:
            print(token.text, [w.text for w in token.words])

INFO:stanza:Downloaded file to /root/.cache/stanza/1.15.0/resources/resources.json
INFO:stanza:Downloading default packages for language: ar (Arabic) ...


models/default.zip: reconstructing file:   0%|          |  0.00B /  459MB            

models/default.zip: downloading bytes:           |  0.00B            

INFO:stanza:Downloaded file to /root/.cache/stanza/1.15.0/resources/ar/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.15.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.15.0/resources/resources.json
INFO:stanza:Loading these models for language: ar (Arabic):
| Processor | Package       |
-----------------------------
| tokenize  | padt_nocharlm |
| mwt       | padt          |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Done loading processors!


وقرأ ['و', 'قرأ']
الولد ['الولد']
الكتاب ['الكتاب']
والد ['و', 'ألد']
الولد ['الولد']


## 8. My own و splitter

Rule: split the و only if the rest of the word appears more than 5 times in the reviews and at least as often as the full word. The threshold of 5 is my own choice.

In [11]:
import re
from collections import Counter

all_tokens = []
for text in df["text"]:
    all_tokens += re.findall(r"[\u0621-\u064A]+", text)

vocab = Counter(all_tokens)

def split_waw(word):
    if word.startswith("و") and len(word) > 3:
        rest = word[1:]
        if vocab[rest] > 5 and vocab[rest] >= vocab[word]:
            return ["و", rest]
    return [word]

words = ["وجبة", "والد", "والله", "والخدمة", "وصلت"]

for w in words:
    print(w, vocab[w], vocab[w[1:]], split_waw(w))

وجبة 267 0 ['وجبة']
والد 0 1 ['والد']
والله 300 692 ['و', 'الله']
والخدمة 137 525 ['و', 'الخدمة']
وصلت 30 0 ['وصلت']


## 9. Summary

**What I tried and what I found**

| Experiment | Result |
|---|---|
| Manual tokenization | `split()` leaves punctuation attached (`'الولد،'`); a regex that keeps only Arabic letters cleans it |
| Arabic vs English | Same meaning: 4 Arabic tokens vs 10 English tokens |
| Mini stemmer | 4 of 5 words correct; `والد` ("father") was wrongly cut to `الد` |
| TF-IDF from scratch | Matches scikit-learn exactly |
| AraBERT v2 on raw text | Broken subwords (`الو` + `##لد`): this model expects Farasa-segmented text |
| AraBERT v02 + preprocessor | Common words stay whole; the و is not separated |
| Farasa | Download failed, and the code silently returned 0 tokens |
| Stanza | Separates the و in `وقرأ`, but turns `والد` into `و` + `ألد` |
| My own و splitter | Splits only if the remainder is common in the data (more than 5 times); correct on 5 test words |

**What I learned**

- The model, the tokenizer, and the preprocessing must match. A mismatch breaks results without any error message.
- No tool separated the و perfectly. `والد` broke in several of them.
- My splitter worked on 5 words only. The threshold of 5 is my own choice, so it is an experiment, not a finished tool.

**In my own words (what a tokenizer does before text reaches the model):**

A tokenizer splits an Arabic sentence into small pieces called tokens.

Then each token is converted into an integer (an ID).

The model only receives integers, not letters or words.